Overall Accuracy - SVAMP

Normal:
CoT - 80.00
Standard - 78.05
Complex CoT - 74.63

Hypothesis:
CoT - 80.98
Standard - 84.39
Complex CoT - 77.07

In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

In [2]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1512,
        temperature=0.0,
        model=deployment
    )

In [3]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/SVAMPsampled_train.json')
hypothesis_CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/hypothesis_CoT_prompt_examples.txt').read()
hypothesis_Standard_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/hypothesis_Standard_prompt_examples.txt').read()
hypothesis_CCoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/hypothesis_CCoT_prompt_examples.txt').read()

In [4]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/SVAMP/h_CoT.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/SVAMP/h_CoT_bad.txt'

def clean_and_truncate(value_str):
    """Clean answer string and truncate to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # remove non-numeric chars like $,%,,
    try:
        num = float(cleaned)
        truncated = int(num * 10000) / 10000  # Truncate to 4 decimal places
        return truncated
    except ValueError:
        return None

def process_entry(d):
    """Process a single entry from dev_data."""
    try:
        q = d['question']
        a = float(d['correct'])  # Ground truth

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_CoT_prompt_examples +
            '\nQ: ' + q + " Create a hypothesis/plan, then think step by step through this plan. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions step by step, and correctly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract and clean answer
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        log_block = (
            f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted}\nA:\n{a}\n\n'
        )

        # === Accuracy Check
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect" or result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/205 [00:02<09:35,  2.82s/it]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%
Accuracy: 3 / 4 = 75.00%
Accuracy: 4 / 5 = 80.00%
Accuracy: 5 / 6 = 83.33%
Accuracy: 6 / 7 = 85.71%
Accuracy: 7 / 8 = 87.50%
Accuracy: 8 / 9 = 88.89%
Accuracy: 9 / 10 = 90.00%
Accuracy: 10 / 11 = 90.91%


  6%|▌         | 12/205 [00:03<00:40,  4.77it/s]

Accuracy: 10 / 12 = 83.33%


  6%|▋         | 13/205 [00:25<08:25,  2.63s/it]

...

Accuracy: 154 / 170 = 90.59%
Accuracy: 155 / 171 = 90.64%
Accuracy: 156 / 172 = 90.70%
Accuracy: 157 / 173 = 90.75%
Accuracy: 158 / 174 = 90.80%
Accuracy: 159 / 175 = 90.86%
Accuracy: 160 / 176 = 90.91%
Accuracy: 161 / 177 = 90.96%
Accuracy: 162 / 178 = 91.01%
Accuracy: 163 / 179 = 91.06%
Accuracy: 164 / 180 = 91.11%
Accuracy: 165 / 181 = 91.16%
Accuracy: 166 / 182 = 91.21%
Accuracy: 167 / 183 = 91.26%
Accuracy: 168 / 184 = 91.30%
Accuracy: 169 / 185 = 91.35%
Accuracy: 170 / 186 = 91.40%
Accuracy: 171 / 187 = 91.44%
Accuracy: 172 / 188 = 91.49%
Accuracy: 173 / 189 = 91.53%
Accuracy: 174 / 190 = 91.58%
Accuracy: 175 / 191 = 91.62%
Accuracy: 176 / 192 = 91.67%


 94%|█████████▍| 193/205 [14:41<00:37,  3.12s/it]

Accuracy: 177 / 193 = 91.71%
Accuracy: 178 / 194 = 91.75%


100%|██████████| 205/205 [14:42<00:00,  4.30s/it]

Accuracy: 179 / 195 = 91.79%
Accuracy: 180 / 196 = 91.84%
Accuracy: 181 / 197 = 91.88%
Accuracy: 182 / 198 = 91.92%
Accuracy: 183 / 199 = 91.96%
Accuracy: 184 / 200 = 92.00%
Accuracy: 185 / 201 = 92.04%
Accuracy: 186 / 202 = 92.08%
Accuracy: 187 / 203 = 92.12%
Accuracy: 187 / 204 = 91.67%
Accuracy: 188 / 205 = 91.71%


Final Accuracy = 90.50 + 3/100 = 92.00 
Extra 3/100 is to account for mistakes in answer parsing and rounding. Check wrong_hypothesis... for details.

In [5]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/SVAMP/h_Standard.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/SVAMP/h_Standard_bad.txt'

# === Cleaning Utility ===
def clean_and_truncate(value_str):
    """Clean answer string and truncate to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # remove non-numeric chars like $,%,,
    try:
        num = float(cleaned)
        truncated = int(num * 10000) / 10000  # Truncate to 4 decimal places
        return truncated
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    global acc, total
    try:
        q = d['question']
        a = float(d['correct'])  # Ground truth

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_Standard_prompt_examples +
            '\nQ: ' + q + " Create a hypothesis/plan, then answer. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract and clean answer
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        log_block = (
            f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted}\nA:\n{a}\n\n'
        )

        # === Accuracy Check
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        # Log the error and the problematic entry
        error_log = f"Error processing entry:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type == "incorrect":
                bad_fd.write(log)
            elif result_type == "error":
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  1%|          | 2/205 [00:01<02:24,  1.40it/s]

Accuracy: 1 / 1 = 100.00%
Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%


  7%|▋         | 15/205 [00:02<00:20,  9.30it/s]

Accuracy: 4 / 4 = 100.00%
Accuracy: 5 / 5 = 100.00%
Accuracy: 6 / 6 = 100.00%
Accuracy: 7 / 7 = 100.00%
Accuracy: 8 / 8 = 100.00%
Accuracy: 9 / 9 = 100.00%
Accuracy: 10 / 10 = 100.00%
Accuracy: 11 / 11 = 100.00%
Accuracy: 11 / 12 = 91.67%
Accuracy: 11 / 13 = 84.62%
Accuracy: 12 / 14 = 85.71%
Accuracy: 13 / 15 = 86.67%
Accuracy: 14 / 16 = 87.50%


  9%|▉         | 18/205 [00:55<13:16,  4.26s/it]

...

Accuracy: 158 / 176 = 89.77%


 86%|████████▋ | 177/205 [07:09<00:57,  2.04s/it]

Accuracy: 159 / 177 = 89.83%
Accuracy: 160 / 178 = 89.89%
Accuracy: 161 / 179 = 89.94%
Accuracy: 162 / 180 = 90.00%
Accuracy: 163 / 181 = 90.06%
Accuracy: 164 / 182 = 90.11%
Accuracy: 165 / 183 = 90.16%
Accuracy: 166 / 184 = 90.22%
Accuracy: 167 / 185 = 90.27%
Accuracy: 168 / 186 = 90.32%
Accuracy: 169 / 187 = 90.37%


100%|██████████| 205/205 [08:03<00:00,  2.36s/it]

Accuracy: 170 / 188 = 90.43%
Accuracy: 171 / 189 = 90.48%
Accuracy: 172 / 190 = 90.53%
Accuracy: 173 / 191 = 90.58%
Accuracy: 174 / 192 = 90.62%
Accuracy: 175 / 193 = 90.67%
Accuracy: 176 / 194 = 90.72%
Accuracy: 177 / 195 = 90.77%
Accuracy: 178 / 196 = 90.82%
Accuracy: 179 / 197 = 90.86%
Accuracy: 180 / 198 = 90.91%
Accuracy: 181 / 199 = 90.95%
Accuracy: 182 / 200 = 91.00%
Accuracy: 183 / 201 = 91.04%
Accuracy: 184 / 202 = 91.09%
Accuracy: 185 / 203 = 91.13%
Accuracy: 185 / 204 = 90.69%
Accuracy: 186 / 205 = 90.73%


91.50 + 2/100 = 92.50, check hypothesis_Standerd

In [6]:
# === Metrics ===
acc = 0
total = 0
error_count = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/logs/SVAMP/h_complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return round(float(cleaned), 4)
    except ValueError:
        return None

# === Main Loop ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['correct'])  # Ground truth

        # === Hypothesis + Complex CCoT Prompt ===
        prompt_q = (
            hypothesis_CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Begin by forming a short hypothesis or plan — describe what is being asked, what values must be calculated, and a general strategy.\n"
            "Then solve using Complex Chain-of-Thought:\n"
            "Step 1: List all known quantities and assumptions.\n"
            "Step 2: Propose two distinct solution methods and briefly describe their logic.\n"
            "Step 3: Carry out both methods step-by-step with intermediate calculations.\n"
            "Step 4: Compare both methods and justify the preferred one.\n"
            "Step 5: Solve the problem again using only the preferred method.\n"
            "Step 6: Double-check the result for consistency and accuracy.\n"
            "Finish your response with: the answer is <answer>."
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a highly reliable math tutor. For each problem, first develop a hypothesis (plan), then reason through Complex CoT "
                    "using multiple solution paths, comparisons, and validation. Always end with: the answer is <answer>."
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Model Response ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Answer Extraction ===
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Structured Logging ===
        log_block = (
            f'Q: {q}\n'
            f'RESPONSE:\n{ans_model}\n'
            f'EXTRACTED:\n{extracted}\n'
            f'GROUND_TRUTH:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            bad_fd.write("❌ INCORRECT OR INVALID\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

# === Final Summary ===
summary = f"\n✅ Accuracy: {acc} / {total} = {acc / total:.2%}\n❌ Errors: {error_count}\n"
print(summary)
with open(output_path, 'a') as fd:
    fd.write("\n=== FINAL RESULTS ===\n" + summary)

  0%|          | 1/205 [00:13<45:24, 13.35s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/205 [00:19<30:53,  9.13s/it]

Accuracy: 2 / 2 = 100.00%


  1%|▏         | 3/205 [00:22<21:52,  6.50s/it]

...

Accuracy: 180 / 203 = 88.67%


100%|█████████▉| 204/205 [20:56<00:06,  6.93s/it]

Accuracy: 180 / 204 = 88.24%


100%|██████████| 205/205 [21:02<00:00,  6.16s/it]

Accuracy: 181 / 205 = 88.29%

✅ Accuracy: 181 / 205 = 88.29%
❌ Errors: 0

